# الاستيعاب المتأخر: الهضبة التي كانت قاعدة التوقّف لتصدّقها

**التأخير رقم يمكن قياسه، والتوقّف المبكر يقع داخله** · معالج رسوميات اختياري · ~45 دقيقة · Colab

تدرّب نموذجاً صغيراً على مهمة يقدر على تعلّمها قطعاً. تصعد دقة التدريب إلى الكمال خلال بضع مئات من الخطوات ثم تتوقف عن الحركة. وتبقى دقة الاختبار عند مستوى الصدفة ولا تبرحه. وكل حدس يقول الشيء ذاته: حفظ النموذج أزواج التدريب، ولا شيء يتحسّن، فأوقف التشغيل وغيّر شيئاً.

والاستيعاب المتأخر هو ما يحدث إن لم تفعل. فبعد آلاف الخطوات — بعد النقطة التي كان ممارس متيقّظ ليتوقف عندها بكثير — تغادر دقة الاختبار مستوى الصدفة وتصعد إلى ما يقارب الكمال. أما أن ذلك يحدث فهو نتيجة الورقة، وقراءتها تكفي لتعلّمه. وما لا تمنحك إياه القراءة هو حجم الفجوة على جهازك أنت، ولا أن قاعدة توقّف مبكر عادية، مطبَّقة على منحناك المسجَّل، تنطلق داخل الفجوة فترمي النتيجة.

### الهدف

درّب محوّلاً صغيراً على الجمع القياسي حتى يحفظ، وواصل التدريب إلى ما بعد تلك النقطة بكثير، وقِس ثلاثة أشياء: الخطوة التي حفظ عندها، والخطوة التي عمّم عندها، والمسافة بينهما. ثم طبّق قاعدة توقّف مبكر عادية على منحناك المسجَّل واحسب الخطوات التي كانت لترميها.

### الأوراق وراء هذه الورشة

- [grokking](https://azimuth.plus/ar/paper/grokking) — تعميم يصل بعد فرط التخصيص بزمن طويل، على مجموعات خوارزمية صغيرة — باور وزملاؤه، 2022

> احفظ نسخة في Drive قبل أن تبدأ (ملف ← حفظ نسخة في Drive). التعديلات على الأصل لا تُحفظ.

## الإعداد

`PROFILE` هو المقبض الوحيد للحجم. المستوى المجاني هو الافتراضي ويعمل داخل حدود Colab المجانية.

In [ ]:
SLUG = "grokking-the-long-plateau"
LANG = "ar"
PROFILE = "free"  # free | a100

# Colab defaults to inline figures; CI does not. Being explicit means the
# captured plot on the site and the plot you see are produced the same way.
%matplotlib inline

# The shim lives in this repository, not on PyPI: a workshop should never
# depend on a package index staying up.
import os
import subprocess
import sys
from pathlib import Path

# Guarded three ways. Colab users re-run the setup cell constantly, and
# a contributor may already be sitting inside a checkout — the first
# Windows run of this notebook cloned the repository into its own
# generated/notebooks/ directory because neither case was handled.
# The hash of the code.py THESE CELLS were built from. setup()
# compares it with the code.py it finds on disk: if a notebook is
# older than its source, every number below describes code that is
# not the code anyone is reading. The printed `code ·` line was
# taken from disk and so could not catch this by itself.
os.environ['AZIMUTH_NOTEBOOK_CODEHASH'] = 'a3a17839dae63a96'

REPO = 'azimuth-workshops'
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'shim' / 'azimuth_nb').is_dir()), None)
if root is None:
    if not Path(REPO).exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', 'stable',
             'https://github.com/MotazSabri/azimuth-workshops.git', REPO],
            check=True,
        )
    root = (here / REPO).resolve()
print('workshop root:', root)

# Absolute, so nothing depends on the working directory. Dropping the
# %cd this cell used to do also means the notebook stops caring where
# it was opened from.
sys.path.insert(0, str(root / 'shim'))
_ = os.environ.setdefault('AZIMUTH_DATA_DIR', str(root / 'data'))

لا شيء هنا غريب. المهمة جمعٌ بقياس عدد أوّلي، والنموذج كتلة محوّل واحدة، والمحسِّن AdamW بحدّ اضمحلال لن تتردد فيه. والورشة كلها تشغيلة تدريب واحدة، تُراقَب أطول مما تُراقَب تشغيلة تدريب عادةً.

اقرأ المنحنى في ثلاثة أجزاء. ترتفع دقة التدريب وتتشبّع، وذلك هو الحفظ. وتجلس الدقة المحجوزة عند مستوى الصدفة زمناً طويلاً، وذلك هو الجزء الذي يُقرأ على أنه فشل. ثم تتحرك. والذي ينبغي أن تنظر إليه وهي مستوية هو خسارة التدريب، فهي تواصل الانخفاض طوال الوقت — التشغيلة لم تنتهِ، بل كفّت عن الإبلاغ عن شيء على المحور الذي كنت تراقبه.

_فحص مسبق. لا يُطلب أي مسرّع — فهذه صغيرة بما يكفي ليكون طلب معالج رسوميات تمثيلاً._

In [ ]:
import azimuth_nb as azimuth

env = azimuth.setup(SLUG, lang=LANG, profile=PROFILE)

_تُعدَّد كل الأزواج التي تسمح بها المهمة، ثم تُقسم مرة واحدة. لاحظ مستوى الصدفة المطبوع هنا واحفظه: فهو الأرضية التي سيجلس عليها المنحنى المحجوز، وهو ليس صفراً._

In [ ]:
import numpy as np

p = env.cfg["modulus"]

# Every pair the task admits, enumerated. This is what makes an algorithmic
# dataset useful here: "unseen" is exact rather than approximate, so the
# held-out number means what it says.
a_all, b_all = np.meshgrid(np.arange(p), np.arange(p), indexing="ij")
a_all, b_all = a_all.ravel(), b_all.ravel()

# Token p is the "=" that separates the operands from the answer position.
# Vocabulary is p operand tokens plus that one; the output layer predicts over
# the p possible answers only.
EQUALS = p
inputs = np.stack([a_all, b_all, np.full_like(a_all, EQUALS)], axis=1)
targets = (a_all + b_all) % p

rng = np.random.default_rng(env.cfg["seed"])
order = rng.permutation(len(inputs))
inputs, targets = inputs[order], targets[order]

split = int(env.cfg["trainFrac"] * len(inputs))
train_in, test_in = inputs[:split], inputs[split:]
train_out, test_out = targets[:split], targets[split:]

n_train, n_test = len(train_in), len(test_in)

# Chance is one over the modulus, NOT zero. A held-out curve resting just
# above the axis is a model guessing uniformly, and mistaking that floor for
# zero makes the eventual jump look larger than it is.
chance = 1.0 / p

if env.lang == "ar":
    print(f"القياس {p} · {len(inputs)} زوجاً ممكناً")
    print(f"تدريب {n_train} · اختبار {n_test}")
    print(f"مستوى الصدفة {chance:.4f}")
else:
    print(f"modulus {p} · {len(inputs)} possible pairs")
    print(f"train {n_train} · test {n_test}")
    print(f"chance level {chance:.4f}")

_كتلة واحدة، صغيرة بما يكفي لقراءتها كاملة. قارن عدد المعاملات بعدد أزواج التدريب أعلاه — فحفظها في المتناول تماماً، وهذا كل سبب حدوث الهضبة الأولى._

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(env.cfg["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEQ_LEN = 3


class GrokFormer(nn.Module):
    """One pre-norm transformer block, reading the answer off the last position.

    Deliberately the smallest thing that groks rather than a scaled-down copy
    of anything. Depth is not the variable here — the training run is.
    """

    def __init__(self, vocab: int, d_model: int, n_heads: int, d_ff: int, n_answers: int):
        super().__init__()
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Parameter(torch.randn(SEQ_LEN, d_model) * 0.02)
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.unembed = nn.Linear(d_model, n_answers, bias=False)

    def forward(self, x):
        h = self.tok(x) + self.pos
        normed = self.ln1(h)
        h = h + self.attn(normed, normed, normed, need_weights=False)[0]
        h = h + self.mlp(self.ln2(h))
        return self.unembed(h[:, -1])


model = GrokFormer(
    vocab=p + 1,
    d_model=env.cfg["dModel"],
    n_heads=env.cfg["nHeads"],
    d_ff=env.cfg["dFF"],
    n_answers=p,
).to(device)

param_count = sum(w.numel() for w in model.parameters())

env.explain("memorization")
if env.lang == "ar":
    print(f"{param_count:,} معامل مقابل {n_train} زوج تدريب")
else:
    print(f"{param_count:,} parameters against {n_train} training pairs")

> **الورقة** · [grokking](https://azimuth.plus/ar/paper/grokking) — تعميم يصل بعد فرط التخصيص بزمن طويل، على مجموعات خوارزمية صغيرة — باور وزملاؤه، 2022
>
> اختار باور وزملاؤه مجموعات خوارزمية صغيرة تحديداً لأن كل مثال فيها قابل للتعداد، فيكون الفصل بين المرئي وغير المرئي دقيقاً وتكفّ الأسئلة عن الحفظ عن كونها تقريبية. وفي ذلك الإطار وُجدت القفزة المتأخرة أولاً، وهو الإطار المُعاد أدناه بقياس أصغر.

> **الحجم** — يعمل الملف المجاني بقياس {{scale.modulus}}، ويتدرب على {{scale.trainFrac}} من الأزواج، ويجري {{scale.steps}} خطوة كاملة الدفعة على معالج مركزي. ويرفع ملف الورقة القياس وميزانية الخطوات. ورفع القياس يجعل المهمة أصعب والتأخير أطول — الهيئة تنتقل، وأعداد الخطوات لا تنتقل.

_الجزء الطويل. راقب الدقتين المطبوعتين تتباعدان ثم تبقيان متباعدتين أطول بكثير مما يبدو معقولاً، وراقب خسارة التدريب تواصل الانخفاض في أثناء ذلك._

In [ ]:
import time

train_x = torch.from_numpy(train_in).long().to(device)
train_y = torch.from_numpy(train_out).long().to(device)
test_x = torch.from_numpy(test_in).long().to(device)
test_y = torch.from_numpy(test_out).long().to(device)

# Full batch. The dataset fits in memory many times over, and a fixed batch
# removes sampling noise from an axis where a long flat line is the evidence.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=env.cfg["learningRate"],
    weight_decay=env.cfg["weightDecay"],
    betas=(0.9, 0.98),
)
criterion = nn.CrossEntropyLoss()


def accuracy(x, y) -> float:
    model.eval()
    with torch.no_grad():
        return float((model(x).argmax(dim=-1) == y).float().mean())


steps_log: list[int] = []
train_acc_log: list[float] = []
test_acc_log: list[float] = []
train_loss_log: list[float] = []

# The two crossings, recorded as they happen rather than reconstructed later:
# the first step at which the model has memorised, and the first at which it
# has generalised. Both stay None if the crossing never occurs, which is a
# distinct state from "crossed at step 0" and must not be flattened into it.
memorise_step = None
generalise_step = None

started = time.time()
model.train()
for step in range(1, env.cfg["steps"] + 1):
    optimizer.zero_grad()
    loss = criterion(model(train_x), train_y)
    loss.backward()
    optimizer.step()
    # .detach() before the scalar conversion: float() on a tensor that still
    # tracks gradients emits a UserWarning, and warnings printed inside a
    # training loop end up on the published page next to the numbers.
    loss_value = loss.detach().item()

    if step == 1 or step % env.cfg["evalEvery"] == 0:
        train_acc = accuracy(train_x, train_y)
        test_acc = accuracy(test_x, test_y)
        model.train()

        steps_log.append(step)
        train_acc_log.append(train_acc)
        test_acc_log.append(test_acc)
        train_loss_log.append(loss_value)

        if memorise_step is None and train_acc >= env.cfg["memoriseAt"]:
            memorise_step = step
        if generalise_step is None and test_acc >= env.cfg["generaliseAt"]:
            generalise_step = step

        if step == 1 or step % env.cfg["logEvery"] == 0:
            elapsed = time.time() - started
            print(
                f"step {step:6d}  train {train_acc:.3f}  test {test_acc:.3f}  "
                f"loss {loss_value:.5f}  {elapsed:5.0f}s"
            )

peak_train_acc = max(train_acc_log)
final_test_acc = test_acc_log[-1]

env.explain("weight decay")
if env.lang == "ar":
    print(f"أعلى دقة تدريب {peak_train_acc:.3f} · دقة الاختبار النهائية {final_test_acc:.3f}")
    print(f"حفظ عند {memorise_step} · عمّم عند {generalise_step}")
else:
    print(f"peak train {peak_train_acc:.3f} · final test {final_test_acc:.3f}")
    print(f"memorised at {memorise_step} · generalised at {generalise_step}")

_محور الدقة يبدأ من الصفر ليبدو المستوي مستوياً، ومحور الخطوات لوغاريتمي لأن التأخير يمتد على رتب من المقدار. انظر إلى اللوحة السفلى بينما لا تفعل العليا شيئاً._

In [ ]:
import matplotlib.pyplot as plt

# A gap of 0 is what an incomplete run produces: either crossing missing means
# there is nothing to measure. Reporting 0 rather than skipping the number
# makes the `delay` check FAIL, which is the honest outcome — a run that never
# grokked should not certify a delay.
if memorise_step is not None and generalise_step is not None:
    grok_gap = generalise_step - memorise_step
else:
    grok_gap = 0

fig, (top, bottom) = plt.subplots(2, 1, figsize=(7, 5.4), sharex=True)

top.plot(steps_log, train_acc_log, label="train", color="#2a9d8f")
top.plot(steps_log, test_acc_log, label="held out", color="#e76f51")
top.axhline(chance, color="#8d99ae", linestyle=":", linewidth=1, label="chance")
# Zero-based: the long flat stretch is the argument, and an autoscaled axis
# would turn the noise on it into a trend.
top.set_ylim(0, 1.02)
top.set_ylabel("accuracy")
top.legend(loc="center left")
top.spines[["top", "right"]].set_visible(False)

bottom.plot(steps_log, train_loss_log, color="#264653")
bottom.set_yscale("log")
bottom.set_ylabel("train loss")
bottom.set_xlabel("optimizer step")
bottom.spines[["top", "right"]].set_visible(False)

for axis in (top, bottom):
    # Logarithmic in the step axis because the delay spans orders of
    # magnitude; on a linear axis the whole first plateau is one pixel wide.
    axis.set_xscale("log")
    if memorise_step is not None:
        axis.axvline(memorise_step, color="#2a9d8f", linewidth=1, alpha=0.5)
    if generalise_step is not None:
        axis.axvline(generalise_step, color="#e76f51", linewidth=1, alpha=0.5)

fig.tight_layout()
plt.show()

env.explain("grokking")
if env.lang == "ar":
    print(f"الفجوة {grok_gap} خطوة بين الحفظ والتعميم")
else:
    print(f"gap of {grok_gap} steps between memorising and generalising")

### تمرين — choose-stopping-rule

اختر صبراً وانظر ما يكلّفك. القاعدة أدناه هي القاعدة العادية: راقب الدقة المحجوزة، وتوقّف حين لا تتحسّن لعدد معطى من التقييمات. ولا شيء يُعاد تدريبه — إنما يُعاد تشغيل المنحنى الذي بين يديك.

اختر الصبر الذي كنت لتستخدمه فعلاً قبل رؤية هذا الشكل، ثم اقرأ الخطوة التي ينطلق عندها والخطوات بينها وبين القفزة. ثم اسأل السؤال الأصعب: ما الذي كان عليك مراقبته بدلاً من ذلك لتنجو القاعدة؟ ثمة مقياس ظل يتحرك خلال الهضبة في هذه التشغيلة، وهو على اللوحة السفلى.

_تلميح متاح: `env.hint(3)`_

In [ ]:
# YOUR TURN.
#
# The ordinary rule: watch held-out accuracy, stop when it has not improved
# for PATIENCE consecutive evaluations. Nothing retrains — this replays the
# curve already logged above.
#
# Patience is counted in EVALUATIONS. Multiply by the eval interval to get the
# number of optimizer steps it actually buys you, which is usually smaller
# than it sounds.
PATIENCE = 20

best_acc = -1.0
stale = 0
stopped_at = steps_log[-1]
for step, acc in zip(steps_log, test_acc_log):
    if acc > best_acc:
        best_acc, stale = acc, 0
    else:
        stale += 1
        if stale >= PATIENCE:
            stopped_at = step
            break

patience_steps = PATIENCE * env.cfg["evalEvery"]
reached = generalise_step if generalise_step is not None else steps_log[-1]
discarded = max(0, reached - stopped_at)

if env.lang == "ar":
    print(f"صبر {PATIENCE} تقييماً = {patience_steps} خطوة")
    print(f"كانت القاعدة لتتوقف عند الخطوة {stopped_at} بدقة محجوزة {best_acc:.3f}")
    print(f"خطوات مهدورة قبل القفزة: {discarded}")
else:
    print(f"patience of {PATIENCE} evaluations = {patience_steps} steps")
    print(f"the rule would have stopped at step {stopped_at}, held-out {best_acc:.3f}")
    print(f"steps discarded before the jump: {discarded}")

_يُفحص الحفظ أولاً وعن قصد: فمن دونه لا هضبة تُدهش منها، وتأخيرٌ مقيسٌ على تشغيلة لم تتعلم شيئاً رقمٌ عن لا شيء._

In [ ]:
# The control goes first. Without memorisation there is no plateau, and a
# delay measured on a run that never fit the training set would be a number
# about nothing.
memorises_ok = env.check("memorises", peak_train_acc)
generalises_ok = env.check("generalises", final_test_acc)
delay_ok = env.check("delay", grok_gap)

In [ ]:
receipt = env.receipt()